# 10.2 · 卷积神经网络 / Convolutional Neural Networks (CNN)

> **课程定位 / Where this fits**
> 第 2 课，**Part 10 · 计算机视觉**。这是 CV 的核心引擎。
> Lesson 2, **Part 10 · Computer Vision**. The core engine of CV.
>
> 10.1 里我们手工设计卷积核做模糊/边缘检测。**CNN 的革命**就是：把卷积核变成**可学习的参数**，让网络从数据里自己学出最有用的核。再配上**池化**和**层层堆叠**，CNN 成为图像任务的统治性架构。本课从"为什么 MLP 处理图像不行"讲起，**从零实现卷积层**，在 FashionMNIST 上训练真 CNN，并**可视化它学到的卷积核与特征图**。
> In 10.1 we hand-designed kernels for blur/edges. **The CNN revolution:** make kernels **learnable parameters** so the network learns the best ones from data. With **pooling** and **stacking**, CNNs dominate image tasks. We start from "why MLPs fail on images," implement a conv layer from scratch, train a real CNN on FashionMNIST, and **visualize its learned kernels and feature maps**.
>
> 💼 **实战/面试视角**："CNN 为什么适合图像 / 参数共享 / 卷积输出尺寸怎么算 / 感受野" 是 CV 面试必考。
> 💼 **Practical/interview angle:** "why CNNs suit images / parameter sharing / output-size formula / receptive field" — CV interview essentials.

> 📐 **符号约定 / Notation**
> - 输入 $(C_{in}, H, W)$，输出 $(C_{out}, H', W')$ / input/output feature maps
> - $K$ 核大小, $S$ 步幅(stride), $P$ 填充(padding) / kernel size, stride, padding
> - 输出尺寸 $H' = \lfloor (H - K + 2P)/S \rfloor + 1$ / output size formula

> 💡 **面试相关 / Interview-relevant**
> - "CNN 三大特性: 局部连接/参数共享/平移不变"（出镜率 ★★★★★）
> - "卷积输出尺寸公式"（★★★★★）
> - "池化的作用"（★★★★）
> - "感受野(receptive field)"（★★★★）
> - "1×1 卷积有什么用"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解为什么 MLP 不适合图像、CNN 强在哪（三大特性）。
   Understand why MLPs fail on images and CNN's three properties.
2. **从零实现卷积层**（含多核、stride、padding），会算输出尺寸。
   Implement a conv layer from scratch (multi-kernel, stride, padding); compute output size.
3. 掌握**池化**与**感受野**。
   Master pooling and receptive fields.
4. 在 FashionMNIST 上训练 CNN，并和 MLP 对比（精度+参数量）。
   Train a CNN on FashionMNIST, compare to an MLP (accuracy + params).
5. **可视化**学到的卷积核与特征图。
   Visualize learned kernels and feature maps.

## 目录 / TOC
1. [为什么 MLP 不适合图像 ⭐](#1)
2. [卷积层：从零实现 ⭐](#2)
3. [池化与感受野 ⭐](#3)
4. [训练真 CNN：CNN vs MLP ⭐](#4)
5. [可视化卷积核与特征图 + 小结 ⭐](#5)


<a id="1"></a>
## 1. 为什么 MLP 不适合图像 ⭐ / Why MLPs Fail on Images

回忆 Part 9 的 MLP：它把图像**拉平成一长串向量**再全连接。这对图像有三个致命问题：
Recall the MLP from Part 9: it **flattens the image into one long vector** then fully connects. Three fatal problems for images:

1. **参数爆炸**：一张 224×224 彩色图拉平是 150528 维，接一个 1000 维隐层就要 **1.5 亿**个权重——单这一层！
   **Parameter explosion:** a 224×224 color image flattens to 150,528 dims; one 1000-unit hidden layer needs **150 million** weights — just one layer!
2. **丢掉空间结构**：拉平后，原本相邻的像素被打散，网络不知道"哪些像素挨在一起"。但图像的意义恰恰在**局部空间关系**（边缘、纹理）。
   **Destroys spatial structure:** flattening scatters neighboring pixels; the net loses "which pixels are adjacent." Yet image meaning lives in **local spatial relations** (edges, textures).
3. **不平移不变**：猫在左上角和在右下角，MLP 要分别用完全不同的权重学两次。
   **Not translation-invariant:** a cat top-left vs bottom-right must be learned twice with entirely different weights.

**CNN 的三大法宝**（面试高频）正好对症：
**CNN's three tricks** (high-frequency interview) cure exactly these:
- **局部连接(local connectivity)**：每个神经元只看一小块区域（核大小），不是整张图。
  **Local connectivity:** each neuron looks at a small patch (kernel size), not the whole image.
- **参数共享(parameter sharing)**：同一个卷积核在整张图上滑动复用——**参数量与图像大小无关**，大幅减少参数。
  **Parameter sharing:** the same kernel slides and is reused across the image — **params independent of image size**, drastically fewer.
- **平移不变(translation invariance)**：核在哪都用同样权重，猫在哪都能被同一个核检测到。
  **Translation invariance:** same weights everywhere, so a cat is detected wherever it is.

下面用参数量对比直观感受。
Let's feel it via a parameter-count comparison.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch, torch.nn as nn
sns.set_theme(style="whitegrid")

H = W = 28; Cin = 1                                    # FashionMNIST 单通道 28×28 / single-channel 28x28
hidden = 128
# MLP 第一层参数: (拉平输入维度) × 隐层 / MLP first-layer params
mlp_params = (H * W * Cin) * hidden + hidden
# 卷积层参数: 核大小 × 输入通道 × 输出通道 + 偏置, 与 H,W 无关! / conv params: independent of H,W
K, Cout = 3, 32
conv_params = (K * K * Cin) * Cout + Cout
print(f"输入图像: {Cin}×{H}×{W}")
print(f"MLP 第一层(→{hidden}隐元):   {mlp_params:,} 个参数")
print(f"卷积层(32个3×3核):          {conv_params:,} 个参数  ← 少了 ~{mlp_params//conv_params} 倍!")
print("\n关键: 卷积参数量只取决于'核大小×通道数', 与图像分辨率无关(参数共享)")
print("→ 同样的核扫全图: 省参数 + 保留空间结构 + 平移不变")


<a id="2"></a>
## 2. 卷积层：从零实现 ⭐ / Conv Layer From Scratch

10.1 我们对**单个核**做过卷积。真正的卷积层有几个关键概念：
In 10.1 we convolved with a **single kernel**. A real conv layer adds key concepts:
- **多个卷积核(filters)**：一层有多个核，每个学一种不同的图案（一个找横边、一个找竖边……），输出**多个特征图(feature maps)**。输出通道数 = 核的个数。
  **Multiple filters:** a layer has many kernels, each learning a different pattern (horizontal edges, vertical edges…), producing **multiple feature maps**. Output channels = number of kernels.
- **步幅(stride $S$)**：核每次滑动几格。$S>1$ 会**下采样**（输出变小）。
  **Stride $S$:** how many cells the kernel jumps. $S>1$ **downsamples** (smaller output).
- **填充(padding $P$)**：在图像边缘补零，让输出尺寸不缩水（或可控）。
  **Padding $P$:** zero-pad the borders to keep output size (or control it).
- **输出尺寸公式(必背)**：$H' = \lfloor (H - K + 2P)/S \rfloor + 1$。
  **Output-size formula (memorize):** $H' = \lfloor (H - K + 2P)/S \rfloor + 1$.

下面**从零实现**带 stride/padding 的卷积，并用一张真实图片把**多个核同时输出的多张特征图**画出来。
Below we implement convolution with stride/padding from scratch, and visualize the **multiple feature maps** from multiple kernels on a real image.


In [ ]:
def conv2d_scratch(img, kernels, stride=1, padding=0):
    """img:(H,W); kernels:(num,kh,kw) → 输出 (num, H', W') / multi-kernel conv with stride+padding."""
    if padding > 0:
        img = np.pad(img, padding, mode="constant")    # 四周补零 / zero-pad borders
    H, W = img.shape; num, kh, kw = kernels.shape
    Ho = (H - kh)//stride + 1; Wo = (W - kw)//stride + 1   # 输出尺寸公式 / output-size formula
    out = np.zeros((num, Ho, Wo))
    for k in range(num):                                # 对每个核 / for each kernel
        for i in range(Ho):
            for j in range(Wo):
                patch = img[i*stride:i*stride+kh, j*stride:j*stride+kw]  # 当前窗口(按stride跳) / window
                out[k, i, j] = np.sum(patch * kernels[k])               # 对应相乘求和 / multiply-sum
    return out

from skimage import data, color, transform
img = transform.resize(color.rgb2gray(data.astronaut()), (64, 64))      # 缩到 64×64 灰度 / 64x64 gray
# 4 个手工核: 横边/竖边/锐化/模糊 (CNN 会自己学出这类核) / 4 hand kernels (CNN learns such kernels)
kernels = np.array([
    [[-1,-1,-1],[0,0,0],[1,1,1]],          # 横向边缘 / horizontal edges
    [[-1,0,1],[-1,0,1],[-1,0,1]],          # 纵向边缘 / vertical edges
    [[0,-1,0],[-1,5,-1],[0,-1,0]],         # 锐化 / sharpen
    np.ones((3,3))/9.0,                     # 模糊 / blur
], dtype=float)
fmaps = conv2d_scratch(img, kernels, stride=1, padding=1)               # 输出4张特征图 / 4 feature maps
print(f"输入 {img.shape} → 4个3×3核, stride=1, padding=1 → 输出特征图 {fmaps.shape}")
print(f"验证输出尺寸: (64-3+2*1)/1+1 = {(64-3+2)//1+1}  ✓")
fig, axes = plt.subplots(1, 5, figsize=(14, 3.2))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("输入"); axes[0].axis("off")
for ax, fm, t in zip(axes[1:], fmaps, ["核1:横边","核2:竖边","核3:锐化","核4:模糊"]):
    ax.imshow(fm, cmap="gray"); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()
print("一层多个核 → 多张特征图, 每张突出一种图案; 这正是 CNN 一个卷积层的输出")


<a id="3"></a>
## 3. 池化与感受野 ⭐ / Pooling & Receptive Field

**池化(pooling)**：把特征图分成小块，每块**只保留一个代表值**——**最大池化(max pooling)** 取最大值（保留最强响应），**平均池化** 取均值。作用：
**Pooling:** divide the feature map into blocks, keep **one summary value** each — **max pooling** takes the max (strongest response), **average pooling** the mean. Purposes:
- **降维**：特征图变小，减少计算和参数。
  **Downsampling:** smaller maps, less compute/params.
- **平移鲁棒**：小幅移动不影响"块内最大值"，让特征更稳定。
  **Translation robustness:** small shifts don't change "block max," stabilizing features.

**感受野(receptive field)**：输出特征图上**一个点，对应原图多大一块区域**。卷积层越堆越深，感受野越大——浅层看局部（边缘），深层看全局（物体）。这解释了 CNN 为何能"逐层从细节到整体"。
**Receptive field:** how large a region of the **original image one output point "sees."** Stacking conv layers grows the receptive field — shallow layers see local (edges), deep layers see global (objects). This is why CNNs go "detail → whole" layer by layer.


In [ ]:
def max_pool(fmap, size=2):
    """2×2 最大池化 / max pooling."""
    H, W = fmap.shape; Ho, Wo = H//size, W//size
    out = np.zeros((Ho, Wo))
    for i in range(Ho):
        for j in range(Wo):
            out[i, j] = fmap[i*size:(i+1)*size, j*size:(j+1)*size].max()  # 每块取最大 / max per block
    return out

edge = fmaps[1]                                       # 用"竖边"特征图演示 / use the vertical-edge map
p1 = max_pool(edge, 2)                                # 一次池化 64→32 / pool once
p2 = max_pool(p1, 2)                                  # 再池化 32→16 / pool again
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, im, t in zip(axes, [edge, p1, p2],
                     [f"特征图 {edge.shape}", f"2×2最大池化 {p1.shape}", f"再池化 {p2.shape}"]):
    ax.imshow(im, cmap="gray"); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()
print("最大池化: 每个小块只留最强响应 → 特征图变小(省算力)+对小位移更鲁棒")
print("感受野: 越深的层, 一个点对应原图越大区域 → 浅层看边缘, 深层看物体整体")


<a id="4"></a>
## 4. 训练真 CNN：CNN vs MLP ⭐ / Training a Real CNN: CNN vs MLP

现在用 PyTorch 在 **FashionMNIST**（10 类服饰灰度图，28×28）上训练一个真正的 CNN，并和参数更多的 MLP 对比。
Now train a real CNN with PyTorch on **FashionMNIST** (10 clothing classes, 28×28 grayscale) and compare to an MLP with more params.

先看看数据长什么样（用前先介绍数据集）。
First, look at the data (introduce the dataset before use).


In [ ]:
import os, torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset

DATA_ROOT = os.path.expanduser("~/.cache/dsfs_cv")     # 共享缓存目录(跨notebook复用) / shared cache
tfm = transforms.ToTensor()                            # 转张量并归一化到[0,1], 形状(1,28,28) / to (1,28,28) in [0,1]
train_full = torchvision.datasets.FashionMNIST(DATA_ROOT, train=True, download=True, transform=tfm)
test_full  = torchvision.datasets.FashionMNIST(DATA_ROOT, train=False, download=True, transform=tfm)
classes = train_full.classes
# 为了训练快, 取子集 / subset for speed
train_ds = Subset(train_full, range(12000)); test_ds = Subset(test_full, range(2000))
print(f"FashionMNIST: 训练全集 {len(train_full)}, 测试全集 {len(test_full)}; 本课用子集 {len(train_ds)}/{len(test_ds)}")
print(f"10 个类别: {classes}")

# 可视化前 10 张图 / show 10 sample images
fig, axes = plt.subplots(2, 5, figsize=(11, 4.5))
for ax, idx in zip(axes.ravel(), range(10)):
    img, lab = train_full[idx]
    ax.imshow(img.squeeze(), cmap="gray"); ax.set_title(classes[lab], fontsize=9); ax.axis("off")
fig.suptitle("FashionMNIST 样例: 28×28 灰度服饰图"); plt.tight_layout(); plt.show()


In [ ]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=256)

# --- MLP: 拉平 → 全连接 / MLP: flatten then dense ---
mlp = nn.Sequential(nn.Flatten(), nn.Linear(28*28, 128), nn.ReLU(), nn.Linear(128, 10))
# --- CNN: 卷积+池化+卷积+池化 → 全连接 / CNN ---
cnn = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # (1,28,28)→(16,14,14)
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # →(32,7,7)
    nn.Flatten(), nn.Linear(32*7*7, 10))                         # 展平接分类头 / flatten + head

def count(m): return sum(p.numel() for p in m.parameters())
def train_eval(model, epochs=5):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); ce = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for xb, yb in train_loader:
            opt.zero_grad(); ce(model(xb), yb).backward(); opt.step()
    model.eval(); correct = total = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            correct += (model(xb).argmax(1) == yb).sum().item(); total += len(yb)
    return correct/total

torch.manual_seed(0)
acc_mlp = train_eval(mlp); acc_cnn = train_eval(cnn)
print(f"MLP:  参数 {count(mlp):,}, test 准确率 = {acc_mlp:.3f}")
print(f"CNN:  参数 {count(cnn):,}, test 准确率 = {acc_cnn:.3f}")
fig, ax = plt.subplots(figsize=(5.5,4))
bars = ax.bar(["MLP","CNN"], [acc_mlp, acc_cnn], color=["#bbb","#39c"])
for b,a in zip(bars,[acc_mlp,acc_cnn]): ax.text(b.get_x()+b.get_width()/2, a+0.005, f"{a:.3f}", ha="center")
ax.set_ylabel("test 准确率"); ax.set_ylim(0.7,1.0); ax.set_title("CNN 用更少/相当参数取得更高精度")
plt.tight_layout(); plt.show()
print("CNN 利用空间结构(局部+参数共享+平移不变) → 比 MLP 更适合图像")


<a id="5"></a>
## 5. 可视化卷积核与特征图 + 小结 ⭐ / Visualizing Kernels & Feature Maps

CNN 不再需要我们手工设计核——它**自己学**出来了。我们把训练好的 CNN **第一层学到的 16 个卷积核**画出来，再看一张测试图经过第一层后的**特征图**。
The CNN no longer needs hand-designed kernels — it **learned them**. Let's visualize the **16 first-layer kernels** the trained CNN learned, then the **feature maps** of a test image after the first layer.


In [ ]:
# 1) 可视化第一层学到的 16 个 3×3 卷积核 / visualize 16 learned 3x3 kernels
W0 = cnn[0].weight.detach().numpy()                   # 形状 (16,1,3,3) / shape (out,in,kh,kw)
fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(W0[i, 0], cmap="RdBu_r"); ax.axis("off")
fig.suptitle("CNN 第一层自己学到的 16 个卷积核(对比10.1手工核)"); plt.tight_layout(); plt.show()

# 2) 一张测试图经过第一个卷积+ReLU 后的特征图 / feature maps after first conv+ReLU
sample, lab = test_full[1]
with torch.no_grad():
    fmap = cnn[1](cnn[0](sample.unsqueeze(0)))         # Conv2d 然后 ReLU / conv then relu
fmap = fmap.squeeze(0).numpy()                         # (16,28,28)
fig, axes = plt.subplots(2, 9, figsize=(14, 3.4))
axes[0,0].imshow(sample.squeeze(), cmap="gray"); axes[0,0].set_title(f"输入:{classes[lab]}", fontsize=8); axes[0,0].axis("off")
axes[1,0].axis("off")
for i in range(16):
    r, c = divmod(i, 8); ax = axes[r, c+1]
    ax.imshow(fmap[i], cmap="viridis"); ax.axis("off")
fig.suptitle("第一卷积层输出的 16 张特征图: 每张突出输入的不同图案"); plt.tight_layout(); plt.show()
print("学到的核 = 各种边缘/纹理检测器(网络自动学, 没人手工设计)")
print("特征图 = 输入对每个核的响应; 不同核激活在不同结构上(边缘/角/区域)")


```
MLP处理图像的问题: 参数爆炸 + 丢空间结构 + 不平移不变
CNN三大特性: 局部连接(只看小块) + 参数共享(核滑动复用,省参数) + 平移不变
卷积层: 多个核→多张特征图; stride下采样; padding保尺寸
输出尺寸: H'=⌊(H-K+2P)/S⌋+1 (必背)
池化: max/avg, 降维+平移鲁棒; 感受野: 越深一个点看原图越大区域(边缘→物体)
CNN自己学卷积核(对比传统CV手工设计); 可视化核=边缘/纹理检测器, 特征图=对核的响应
```

### 💡 面试速查 / Interview cheat-sheet
1. **CNN 三特性**: 局部连接 / 参数共享 / 平移不变。
   CNN's three: local connectivity / parameter sharing / translation invariance.
2. **输出尺寸**: H'=(H−K+2P)/S+1。
   Output size: H'=(H−K+2P)/S+1.
3. **参数共享**: 卷积参数量与图像分辨率无关, 远少于 MLP。
   Parameter sharing: conv params independent of resolution, far fewer than MLP.
4. **池化**: 降维 + 平移鲁棒; max 最常用。
   Pooling: downsample + shift-robust; max most common.
5. **感受野**: 越深看得越广; 浅层边缘, 深层物体。
   Receptive field: deeper sees wider; shallow edges, deep objects.

### 下一节 / Next
**10.3 经典 CNN 架构**——从 LeNet(1998) 到 AlexNet、VGG、GoogLeNet，看 CNN 架构如何一步步演进，理解 1×1 卷积、深度可分离卷积等关键设计，并亲手搭建 LeNet。
**10.3 Classic CNN Architectures** — from LeNet (1998) to AlexNet, VGG, GoogLeNet; how architectures evolved, key designs like 1×1 convolutions, and building LeNet by hand.
